# Практика: восстановление коэффициентов

Решаем задачу [«Восстановление коэффициентов»](https://coderun.yandex.ru/problem/coefficients-restoration) с CodeRun градиентным спуском.

Есть 1000 точек $(x_i, y_i)$, где

$$y = \bigl((a + \varepsilon_a)\sin x + (b + \varepsilon_b)\ln x\bigr)^2 + (c + \varepsilon_c)\,x^2, \qquad \varepsilon \in [-0{,}001;\ 0{,}001].$$

Числа $a, b, c > 0$ неизвестны. Нужно найти их с точностью $10^{-2}$.

**Как работать с ноутбуком.** Ячейки с `...` заполняете вы. После каждого задания есть ячейка с проверкой. Застряли — раскройте подсказку. Сначала подсказка, решение — только если совсем не идёт.

## План

1. Данные и модель.
2. Функция ошибки.
3. Производные по $a$, $b$, $c$ на бумаге.
4. Градиент в коде и его проверка.
5. Градиентный спуск: сначала на данных с известным ответом, потом на настоящих.
6. Эксперименты с шагом.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["font.size"] = 12

## 1. Данные

Скачайте `data.csv` со страницы задачи и положите рядом с ноутбуком. В файле два столбца без заголовка: $x$ и $y$.

Кроме настоящих данных сделаем **демо-данные** по той же формуле, но с известными коэффициентами $a = 1{,}2$, $b = 2$, $c = 3$. На них удобно проверять код: правильный ответ мы знаем заранее.

In [ ]:
def make_demo_data(a, b, c, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.01, 10, n)
    noise = lambda: rng.uniform(-0.001, 0.001, n)
    y = ((a + noise()) * np.sin(x) + (b + noise()) * np.log(x)) ** 2 + (c + noise()) * x**2
    return x, y


x_demo, y_demo = make_demo_data(1.2, 2, 3)

if Path("data.csv").exists():
    data = np.loadtxt("data.csv", delimiter=",")
    x, y = data[:, 0], data[:, 1]
    print("Загружен data.csv:", len(x), "точек")
else:
    x, y = x_demo, y_demo
    print("data.csv не найден, дальше работаем с демо-данными")

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x, y, s=6, alpha=0.6)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Данные")
plt.show()

### Вопрос

Что в этой задаче **данные**, а что **параметры**, которые мы подбираем? От чего зависит функция ошибки?

<details>
<summary><b>Ответ</b></summary>

Данные — числа $x_i$, $y_i$: они заданы и не меняются. Параметры — $a$, $b$, $c$.

Функция ошибки зависит только от параметров: $L(a, b, c)$. Её мы и будем минимизировать. Значения $\sin x_i$, $\ln x_i$, $x_i^2$ можно посчитать один раз — это просто числа.

</details>

## 2. Модель и ошибка

Шум $\varepsilon$ маленький, поэтому модель берём без него:

$$\hat y = (a\sin x + b\ln x)^2 + c\,x^2.$$

Ошибка — среднеквадратичная:

$$L(a, b, c) = \frac1n\sum_{i=1}^n (\hat y_i - y_i)^2.$$

### Задание 1

Допишите `predict` и `mse`. Работайте сразу с массивами numpy, без циклов.

In [ ]:
def predict(x, a, b, c):
    return ...  # ваш код


def mse(x, y, a, b, c):
    return ...  # ваш код

<details>
<summary><b>💡 Подсказка</b></summary>

В numpy есть `np.sin`, `np.log` (натуральный логарифм) и `np.mean`. Они работают поэлементно: `np.sin(x)` — массив синусов всех $x_i$.

</details>

<details>
<summary><b>✅ Решение</b></summary>

```python
def predict(x, a, b, c):
    return (a * np.sin(x) + b * np.log(x)) ** 2 + c * x**2


def mse(x, y, a, b, c):
    return np.mean((predict(x, a, b, c) - y) ** 2)
```

</details>

In [ ]:
# Проверка
assert np.isclose(predict(np.array([1.0]), 1, 1, 1)[0], np.sin(1) ** 2 + 1)
assert mse(x_demo, y_demo, 1.2, 2, 3) < 0.01
print("Ошибка в стартовой точке a = b = c = 1:", mse(x, y, 1, 1, 1))
print("Всё верно!")

## 3. Производные на бумаге

Введём обозначения для одного объекта:

$$s_i = a\sin x_i + b\ln x_i, \qquad \mathrm{err}_i = \hat y_i - y_i = s_i^2 + c\,x_i^2 - y_i.$$

Тогда $L = \frac1n\sum_i \mathrm{err}_i^2$.

### Задание 2

Найдите $\dfrac{\partial L}{\partial a}$, $\dfrac{\partial L}{\partial b}$, $\dfrac{\partial L}{\partial c}$. Запишите ответ через $\mathrm{err}_i$, $s_i$ и $x_i$.

<details>
<summary><b>💡 Подсказка 1</b></summary>

Производная суммы — сумма производных, поэтому достаточно продифференцировать один член $\mathrm{err}_i^2$.

Цепное правило: внешняя функция — квадрат, внутренняя — $\mathrm{err}_i$:

$$\frac{\partial}{\partial a}\mathrm{err}_i^2 = 2\,\mathrm{err}_i\cdot\frac{\partial\,\mathrm{err}_i}{\partial a}.$$

</details>

<details>
<summary><b>💡 Подсказка 2</b></summary>

В $\mathrm{err}_i$ от $a$ зависит только $s_i^2$. Ещё раз цепное правило:

$$\frac{\partial\,\mathrm{err}_i}{\partial a} = 2 s_i\cdot\frac{\partial s_i}{\partial a} = 2 s_i\sin x_i.$$

По $c$ проще: $\dfrac{\partial\,\mathrm{err}_i}{\partial c} = x_i^2$.

</details>

<details>
<summary><b>✅ Ответ</b></summary>

$$\frac{\partial L}{\partial a} = \frac4n\sum_i \mathrm{err}_i\, s_i \sin x_i, \qquad
\frac{\partial L}{\partial b} = \frac4n\sum_i \mathrm{err}_i\, s_i \ln x_i, \qquad
\frac{\partial L}{\partial c} = \frac2n\sum_i \mathrm{err}_i\, x_i^2.$$

</details>

## 4. Градиент в коде

### Задание 3

Допишите `gradient`: функция возвращает массив из трёх чисел $\left(\frac{\partial L}{\partial a}, \frac{\partial L}{\partial b}, \frac{\partial L}{\partial c}\right)$.

In [ ]:
def gradient(x, y, a, b, c):
    s = ...    # ваш код
    err = ...  # ваш код
    grad_a = ...
    grad_b = ...
    grad_c = ...
    return np.array([grad_a, grad_b, grad_c])

<details>
<summary><b>💡 Подсказка</b></summary>

$\frac1n\sum_i$ — это `np.mean`. Например, $\frac{\partial L}{\partial c}$ = `2 * np.mean(err * x**2)`.

</details>

<details>
<summary><b>✅ Решение</b></summary>

```python
def gradient(x, y, a, b, c):
    s = a * np.sin(x) + b * np.log(x)
    err = s**2 + c * x**2 - y
    grad_a = 4 * np.mean(err * s * np.sin(x))
    grad_b = 4 * np.mean(err * s * np.log(x))
    grad_c = 2 * np.mean(err * x**2)
    return np.array([grad_a, grad_b, grad_c])
```

</details>

### Проверка градиента

Ошибиться в формуле легко, поэтому сравним с численной производной из лекции:

$$\frac{\partial L}{\partial a} \approx \frac{L(a + h, b, c) - L(a - h, b, c)}{2h}.$$

Если формулы верны, числа совпадут почти до последнего знака.

In [ ]:
def numeric_gradient(x, y, a, b, c, h=1e-6):
    return np.array([
        (mse(x, y, a + h, b, c) - mse(x, y, a - h, b, c)) / (2 * h),
        (mse(x, y, a, b + h, c) - mse(x, y, a, b - h, c)) / (2 * h),
        (mse(x, y, a, b, c + h) - mse(x, y, a, b, c - h)) / (2 * h),
    ])


point = (0.5, 1.5, 2.0)
print("Формула:  ", gradient(x_demo, y_demo, *point))
print("Численно: ", numeric_gradient(x_demo, y_demo, *point))
assert np.allclose(gradient(x_demo, y_demo, *point), numeric_gradient(x_demo, y_demo, *point), rtol=1e-4)
print("Градиент верный!")

## 5. Градиентный спуск

$$(a, b, c) \leftarrow (a, b, c) - \eta\,\nabla L(a, b, c)$$

### Задание 4

Допишите `fit`: стартуем из $a = b = c = 1$, делаем `steps` шагов с шагом `lr`. В `history` сохраняйте ошибку на каждом шаге — потом нарисуем график.

In [ ]:
def fit(x, y, lr=1e-4, steps=5000, start=(1.0, 1.0, 1.0)):
    w = np.array(start)
    history = []
    for step in range(steps):
        ...  # ваш код: посчитать градиент и сделать шаг
        history.append(mse(x, y, *w))
    return w, history

<details>
<summary><b>💡 Подсказка</b></summary>

`w` — массив `[a, b, c]`. Запись `gradient(x, y, *w)` распаковывает его в три аргумента. Шаг ГД для всех параметров сразу: `w = w - lr * grad`.

</details>

<details>
<summary><b>✅ Решение</b></summary>

```python
def fit(x, y, lr=1e-4, steps=5000, start=(1.0, 1.0, 1.0)):
    w = np.array(start)
    history = []
    for step in range(steps):
        grad = gradient(x, y, *w)
        w = w - lr * grad
        history.append(mse(x, y, *w))
    return w, history
```

</details>

### Сначала — данные с известным ответом

Должно получиться $a \approx 1{,}2$, $b \approx 2$, $c \approx 3$.

In [ ]:
w_demo, history_demo = fit(x_demo, y_demo)
print("Нашли:   ", np.round(w_demo, 3))
print("Ожидали: ", [1.2, 2, 3])
assert np.allclose(w_demo, [1.2, 2, 3], atol=0.01)
print("Работает!")

In [ ]:
fig, ax = plt.subplots()
ax.plot(history_demo)
ax.set_yscale("log")
ax.set_xlabel("шаг")
ax.set_ylabel("MSE")
ax.set_title("Ошибка на демо-данных")
plt.show()

### Теперь — настоящие данные

Запустите `fit` на `x`, `y` и посмотрите на ответ и график ошибки.

In [ ]:
w, history = fit(x, y)
a, b, c = w
print("MSE в конце:", history[-1])
print(f"{a:.2f} {b:.2f} {c:.2f}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(history)
ax.set_yscale("log")
ax.set_xlabel("шаг")
ax.set_ylabel("MSE")
ax.set_title("Ошибка на data.csv")
plt.show()

Последняя строка — ответ для CodeRun: три числа через пробел, два знака после запятой. Отправьте его и проверьте!

## 6. Эксперименты с шагом

### Задание 5

Запустите `fit` с шагами $10^{-3}$, $3\cdot10^{-4}$, $10^{-4}$, $3\cdot10^{-5}$ и нарисуйте ошибку для каждого. Какой шаг лучший? Что происходит при $10^{-3}$?

In [ ]:
fig, ax = plt.subplots()
for lr in [1e-3, 3e-4, 1e-4, 3e-5]:
    with np.errstate(all="ignore"):
        w_lr, history_lr = fit(x, y, lr=lr, steps=3000)
    ax.plot(history_lr, label=f"lr = {lr:g}")
    print(f"lr = {lr:g}: MSE в конце = {history_lr[-1]:.3g}")
ax.set_yscale("log")
ax.set_xlabel("шаг")
ax.set_ylabel("MSE")
ax.legend()
plt.show()

<details>
<summary><b>🤔 Почему при 10⁻³ всё ломается?</b></summary>

Ошибка улетает в бесконечность (в numpy это `inf` и `nan`): шаг слишком большой, и каждый шаг перепрыгивает минимум всё дальше — как на 3D-картинке с лекции.

Виноват масштаб: $x$ доходит до 10, значит $x^2$ — до 100, а в $\frac{\partial L}{\partial c}$ стоит множитель $x_i^2$. Градиент по $c$ огромный, и шаг приходится подбирать под него. Это тот же эффект, что с ненормализованными признаками на лекции.

При $3\cdot10^{-5}$ спуск устойчивый, но медленный: за 3000 шагов точности может не хватить.

</details>

## Бонус

### 1. Зачем в условии $a, b > 0$?

Запустите `fit` из точки `start=(-1.0, -1.0, 1.0)`. Что получилось и почему это тоже правильный минимум?

In [ ]:
w_neg, history_neg = fit(x, y, start=(-1.0, -1.0, 1.0))
print(np.round(w_neg, 3), "MSE:", history_neg[-1])

<details>
<summary><b>Ответ</b></summary>

Получаются $-a$ и $-b$ с той же ошибкой. Внутри квадрата знак не важен:

$$(-a\sin x - b\ln x)^2 = (a\sin x + b\ln x)^2.$$

У функции ошибки два одинаково глубоких минимума — функция невыпуклая. Куда придёт спуск, зависит от старта. Условие $a, b > 0$ выбирает один из ответов, а старт из $a = b = 1$ приводит именно к нему.

</details>


### 2. Почему ошибка не падает до нуля?

<details>
<summary><b>Ответ</b></summary>

В данных есть шум $\varepsilon$: коэффициенты в каждой точке чуть-чуть разные. Никакие $a, b, c$ не опишут все точки точно, поэтому MSE останавливается на маленьком, но ненулевом значении. Модель нашла коэффициенты «в среднем» — именно их и нужно найти в задаче.

</details>